# Label quality — what can the labels support?

| | |
|---|---|
| **in** | DrEval's reprocessed CTRPv2 table (per-curve fit statistics), `outputs/panel/*` |
| **out** | `outputs/diagnostics/label_quality_vs_performance.csv` |

**Why.** The target is not a measurement but a **parameter of a fitted curve**. DrEval re-fit every
dose-response curve with CurveCurator, folding replicate variability into the fit rather than
averaging replicates beforehand. Those fits carry quality statistics — `R2`, `RMSE`, `pValue` — that
travel with every label and that nothing in this project read until 14.08.2026.

**Why it replaces the replicate-noise route.** Replicate disagreement cannot be measured on this
target: one fit per (cell line, drug), no pair left to disagree. It would also reach only 6 of this
project's 181 cell lines. Fit quality covers **all** of them.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUT = ROOT / 'notebooks' / 'outputs'
DIAG, PANEL = OUT / 'diagnostics', OUT / 'panel'

def run(script):
    """Delegate to the committed script rather than restating the experiment here."""
    import subprocess
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'evaluation' / script)], check=True)

RUN_LABEL_QUALITY = False       # cheap (reads a CSV), but the script is the definition
print(f'RUN_LABEL_QUALITY = {RUN_LABEL_QUALITY}')

RUN_LABEL_QUALITY = False


## 1 · How good are the fits behind our labels?

In [2]:
if RUN_LABEL_QUALITY:
    run('label_quality.py')
q = pd.read_csv(DIAG / 'label_quality_vs_performance.csv')
print(q.round(4).to_string(index=False))
print(f"\ndrugs: {len(q)}   curves behind them: {int(q['n'].sum()):,}")

       drug  median_R2  frac_ns  label_sd   n  X_pca  X_scGPT
 crizotinib     0.9668   0.0000    0.0891 179 0.1756  -0.0139
doxorubicin     0.9567   0.0055    0.1329 182 0.3196   0.2272
   afatinib     0.9207   0.0118    0.1337 169 0.4637   0.4179
  dasatinib     0.8789   0.0164    0.1686 183 0.5134   0.5037
  erlotinib     0.8760   0.0442    0.1218 181 0.4371   0.4386
  etoposide     0.8910   0.0449    0.0896 178 0.2357   0.1072
 paclitaxel     0.5829   0.0452    0.2022 177 0.2539   0.2265
  sorafenib     0.9063   0.0730    0.0960 178 0.1266   0.1460
gemcitabine     0.8155   0.0929    0.2045 183 0.1482   0.0951
   imatinib     0.6905   0.1271    0.0857 181 0.0177  -0.0292
     platin     0.1665   0.7333    0.0617 180 0.0295   0.0909

drugs: 11   curves behind them: 1,971


## 2 · Does label quality predict per-drug performance?

If it does, label quality is a live constraint on results rather than a hypothesis about one.

In [3]:
from scipy.stats import spearmanr
for rep in ['X_pca', 'X_scGPT']:
    for metric, sense in [('median_R2', 'higher = better fit'), ('frac_ns', 'higher = worse fit')]:
        r = spearmanr(q[metric], q[rep])
        print(f'  {rep:8s} vs {metric:10s} ({sense:19s}): rho={r.statistic:+.3f}  p={r.pvalue:.3f}')
r2 = spearmanr(q.frac_ns, q.label_sd)
print(f'\nCONTROL -- is bad fit confounded with low label spread?  rho={r2.statistic:+.3f} p={r2.pvalue:.3f}')
print('  -> uncorrelated, so fit quality and label spread are SEPARATE drivers')

  X_pca    vs median_R2  (higher = better fit): rho=+0.373  p=0.259
  X_pca    vs frac_ns    (higher = worse fit ): rho=-0.718  p=0.013
  X_scGPT  vs median_R2  (higher = better fit): rho=+0.200  p=0.555
  X_scGPT  vs frac_ns    (higher = worse fit ): rho=-0.464  p=0.151

CONTROL -- is bad fit confounded with low label spread?  rho=-0.227 p=0.502
  -> uncorrelated, so fit quality and label spread are SEPARATE drivers


## What this settles

**Label quality is measured and it predicts performance.** Roughly one label in nine comes from a
curve that does not establish a dose-response, and the per-drug rate of such fits predicts out-of-fold
Spearman. Label spread predicts it independently, and the two are uncorrelated.

⚠️ **n = 11 drugs.** Indicative of a real effect, not a precise estimate of its size.

⚠️ **What is still not quantified:** the share of residual error that is *irreducible*. Fit quality
bounds the labels; it does not give the noise floor directly.